This notebook trains 2 machine learning models.

The preprocessing of the transcription data includes:
- Run the spaCy pipeline that includes **named entity recognition**
- Carry out lemmatization and remove clinically irrelevant stop words 

The ML pipeline is structured like this:
- Split the data into a training and testing dataset.
- Have a cross validation of a linear SVC pipeline and a multinomial logistic regression
- Determine the model with the best test matrix and fit it on the training dataset.

In [1]:
# Import the relevant libraries for preprocessing
import pandas as pd
import spacy

In [2]:
# Load the appropriate data
super_clinical_mtsamples = pd.read_csv("../data/super_clinical_mtsamples.csv")
super_clinical_mtsamples.head()

,description,medical_specialty,sample_name,transcription,keywords,medical_specialty_clean,clinical_superclass
0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller...",Allergy / Immunology,internal_and_systemic_medicine
1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh...",Bariatrics,surgical_and_procedural
2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart...",Bariatrics,surgical_and_procedural
3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple...",Cardiovascular / Pulmonary,internal_and_systemic_medicine
4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo...",Cardiovascular / Pulmonary,internal_and_systemic_medicine


In [3]:
# Load the spaCy model
# nlp = spacy.load("en_core_sci_sm") # The clinical dataset is not available for download
nlp = spacy.load("en_core_web_sm")

# Load the clinical stopwords
with open('../data/clinical-stopwords.txt', 'r') as file:
    custom_stopwords = set(file.read().splitlines())

# Create a function that carries out the preprocessing
def extract_and_lemmatize_entities(text):
    # Handle missing/NaN transcripts gracefully
    if not isinstance(text, str):
        return ""
        
    doc = nlp(text)
    processed_entities = []
    
    # Iterate ONLY through the recognized named entities
    for ent in doc.ents:
        # Lemmatize and filter out custom stopwords and punctuation
        ent_tokens = [
            token.lemma_.lower() for token in ent 
            if token.lemma_.lower() not in custom_stopwords 
            and not token.is_punct
            and not token.is_space
        ]
        
        # If tokens remain after filtering, join them back into a string
        if ent_tokens:
            processed_entities.append(" ".join(ent_tokens))
            
    # Return a single string of all processed entities for TF-IDF
    return " ".join(processed_entities)

In [4]:
# Compute the entities using the above function
super_clinical_mtsamples['entity_features'] = super_clinical_mtsamples['transcription'].apply(extract_and_lemmatize_entities)

In [5]:
# Saving the data with entities to avoid rerunning the processing
super_clinical_mtsamples.to_pickle("../.private/super_clinical_mtsamples_preprocessed.pkl")

In [6]:
# Load the data that is preprocessed
super_clinical_mtsamples = pd.read_pickle("../.private/super_clinical_mtsamples_preprocessed.pkl")

## Carry out the machine learning pipeline

This pipeline carries out the linearSVC and multinomial logistic regression while creating a cross validation 5 times

In [ ]:
# Load the required libraries for this section
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


In [8]:
# Split the data into a test and split section
# Define Features (X) and Target (y)
X = super_clinical_mtsamples['entity_features']
y = super_clinical_mtsamples['clinical_superclass']

# Train/Test Split (30% test size, random_state 42, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.30, 
    random_state=42, 
    stratify=y
)

In [9]:
# Define the pipeline 
# LinearSVC Pipeline
svc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
                    ngram_range=(1, 2),      
                    min_df=3,                
                    max_df=0.9,               
                    max_features=20000,       
                    sublinear_tf=True,    
                    strip_accents='unicode',
                    norm='l2'
    )),
    ('clf', LinearSVC(class_weight='balanced', random_state=42))
])

# Multinomial Logistic Regression Pipeline
logreg_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),      
            min_df=3,                
            max_df=0.9,               
            max_features=20000,       
            sublinear_tf=True,    
            strip_accents='unicode',
            norm='l2'
    )),
    ('clf', LogisticRegression(
        class_weight='balanced',
        random_state=42,
        max_iter=1000  # increased to help convergence with TF-IDF's high-dimensional sparse features
    ))
])

In [10]:
# Perform the cross validation on the training dataset
print("Running 5-fold CV for LinearSVC...")
svc_cv_results = cross_validate(
    svc_pipeline, X_train, y_train, 
    cv=5, 
    scoring=['accuracy', 'f1_macro']
)

print("Running 5-fold CV for Logistic Regression...")
logreg_cv_results = cross_validate(
    logreg_pipeline, X_train, y_train, 
    cv=5, 
    scoring=['accuracy', 'f1_macro']
)

Running 5-fold CV for LinearSVC...
Running 5-fold CV for Logistic Regression...


In [11]:
# Print training metrics
print("\n--- Cross-Validation Results ---")
print(f"LinearSVC Mean F1-Macro: {np.mean(svc_cv_results['test_f1_macro']):.4f}")
print(f"LogReg Mean F1-Macro:    {np.mean(logreg_cv_results['test_f1_macro']):.4f}")


--- Cross-Validation Results ---
LinearSVC Mean F1-Macro: 0.3983
LogReg Mean F1-Macro:    0.4815


In [15]:
# Fit the better pipeline on the full training set
logreg_pipeline.fit(X_train, y_train)

# Generate predictions on the unseen test set
y_pred = logreg_pipeline.predict(X_test)

# 3. Calculate the requested metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_f1_macro = f1_score(y_test, y_pred, average='macro')

# 4. Print the final results
print("\n--- Test Set Evaluation ---")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"F1-Macro: {test_f1_macro:.4f}")

print("\n--- Detailed Classification Report ---")
# The classification report provides precision, recall, and f1-score per class
print(classification_report(y_test, y_pred))


--- Test Set Evaluation ---
Accuracy: 0.5963
F1-Macro: 0.4756

--- Detailed Classification Report ---
                                precision    recall  f1-score   support

      diagnostic_and_pathology       0.28      0.43      0.33        87
internal_and_systemic_medicine       0.58      0.64      0.61       356
     neurology_and_behavioural       0.31      0.40      0.35        83
      rehab_and_applied_health       0.32      0.46      0.38        56
       surgical_and_procedural       0.81      0.64      0.71       622

                      accuracy                           0.60      1204
                     macro avg       0.46      0.51      0.48      1204
                  weighted avg       0.64      0.60      0.61      1204

